# Tool-Calling Research Agent (Teacher)

**Phase 1** of the BBB conference demo pipeline.

This notebook implements a tool-calling agent using the **OpenAI Responses API**
with reasoning support (gpt-5.4). The agent researches a given stock ticker by
calling financial tools (yfinance), then produces a structured research memo.

The full conversation trajectory (including reasoning summaries and all tool calls)
is saved as training data for fine-tuning Qwen3-4B.

## Architecture
```
User prompt → GPT-5.4 (Responses API)
                ├── reasoning item (summary captured)
                └── function_call item → execute tool → function_call_output → loop
                                                                                 ↓
                                                                         final message
```

## Setup

In [1]:
import json
import os
import sys
from pathlib import Path

from openai import OpenAI
from dotenv import load_dotenv

# Add project root to path so we can import tools
PROJECT_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(PROJECT_ROOT))

from tools.stock_tools import (
    TOOL_SCHEMAS,
    TOOL_FUNCTIONS,
    run_tool_calling_agent,
)

load_dotenv(PROJECT_ROOT / ".env")

client = OpenAI(base_url="https://us.api.openai.com/v1")

# Teacher model — change to gpt-5.4 when available
MODEL = "gpt-5.4"

## System Prompt

Uses the `developer` role (Responses API equivalent of `system`).
Instructs the model to use tools for research and produce a structured memo.
Reasoning happens via the model's internal reasoning tokens — no fake `<think>` tags needed.

In [14]:
SYSTEM_PROMPT = """\
You are an equity research analyst conducting comprehensive research on a given stock.

## Instructions
- Use the available tools to gather financial data, news, price history, and analyst recommendations.
- Before each tool call, take time to think about the task at hand.
- After gathering sufficient data, synthesize your findings into a structured research memo.
- Be thorough but efficient — aim for just the necessary amount of tool calls.

## Output Format
Your final response should be a markdown research memo with these sections:
- **Company Overview**: Brief description and market position
- **Financial Analysis**: Revenue, profitability, balance sheet highlights
- **Market Performance**: Price trends, volatility, trading patterns
- **Analyst Sentiment**: Consensus recommendations, recent upgrades/downgrades
- **Key Risks & Opportunities**: Summary of bull/bear case

## Important
- Use ONLY the tools provided. Do not make up financial data.
- Present facts objectively. Do not give investment advice.
- If a tool returns an error, note it and move on.
"""

## Tool Schemas

These are defined in `tools/stock_tools.py` and shared across all BBB notebooks.

In [ ]:
# Quick look at what tools the agent has access to
# Responses API uses flat format: name/description/parameters at top level
for tool in TOOL_SCHEMAS:
    params = list(tool["parameters"]["properties"].keys())
    print(f"  {tool['name']}({', '.join(params)})")
    print(f"    → {tool['description']}\n")

## Run the Agent

Uses `client.responses.create()` with reasoning enabled.
The agent loop handles:
- Parsing `function_call` items from `response.output`
- Executing tools and sending `function_call_output` back
- Passing reasoning items between turns (required for reasoning models)
- Capturing reasoning summaries for inspection

In [ ]:
result = run_tool_calling_agent(
    client=client,
    model=MODEL,
    user_prompt="Research NVDA focusing on financial health and growth potential.",
    system_prompt=SYSTEM_PROMPT,
    reasoning_effort="medium",
)

print(f"\nTotal input items: {len(result['input'])}")
print(f"Reasoning summaries: {len(result['reasoning_summaries'])}")
print(f"Tokens — input: {result['usage']['input_tokens']}, output: {result['usage']['output_tokens']}, reasoning: {result['usage']['reasoning_tokens']}")

### Inspect the trajectory

The Responses API returns a mix of dict items (our inputs) and SDK objects
(reasoning items, function_call items). Let's trace what happened.

In [ ]:
for i, item in enumerate(result["input"]):
    # Dict items (our input messages and function_call_outputs)
    if isinstance(item, dict):
        role = item.get("role", item.get("type", "?"))
        content = item.get("content", item.get("output", ""))
        if role == "developer":
            print(f"[{i}] DEVELOPER: {str(content)[:80]}...")
        elif role == "user":
            print(f"[{i}] USER: {content}")
        elif item.get("type") == "function_call_output":
            print(f"[{i}] TOOL RESULT ({item.get('call_id', '?')[:15]}): {str(content)[:80]}...")
        else:
            print(f"[{i}] {role}: {str(content)[:80]}...")
    # SDK response objects (reasoning items, function_call items)
    else:
        item_type = getattr(item, "type", "unknown")
        if item_type == "reasoning":
            summary = ""
            if getattr(item, "summary", None):
                summary = item.summary[0].text[:100] if item.summary else ""
            print(f"[{i}] REASONING: {summary}...")
        elif item_type == "function_call":
            print(f"[{i}] FUNCTION CALL: {item.name}({item.arguments[:60]})")
        elif item_type == "message":
            text = ""
            if hasattr(item, "content") and item.content:
                text = item.content[0].text[:100] if item.content else ""
            print(f"[{i}] MESSAGE: {text}...")
        else:
            print(f"[{i}] {item_type}: {str(item)[:80]}")

### View reasoning summaries

These are summaries of the model's *actual* internal reasoning tokens —
not fake `<think>` tags, but real chain-of-thought captured via `reasoning.summary: "auto"`.

In [ ]:
if result["reasoning_summaries"]:
    for i, summary in enumerate(result["reasoning_summaries"]):
        print(f"--- Reasoning step {i+1} ---")
        print(summary)
        print()
else:
    print("No reasoning summaries (model may not support them at this effort level)")

### View the final research memo

In [ ]:
from IPython.display import Markdown, display

if result["output_text"]:
    display(Markdown(result["output_text"]))
else:
    print("No final text output — agent may have hit max iterations")

## Save the Trajectory

We serialize the input list for training data. Responses API items
that are SDK objects get serialized via their `.model_dump()` method.

In [ ]:
def serialize_input_list(input_list: list) -> list[dict]:
    """Convert a mix of dicts and SDK response objects to JSON-serializable dicts."""
    serialized = []
    for item in input_list:
        if isinstance(item, dict):
            serialized.append(item)
        elif hasattr(item, "model_dump"):
            serialized.append(item.model_dump())
        else:
            serialized.append({"type": "unknown", "data": str(item)})
    return serialized


output_dir = PROJECT_ROOT / "data" / "bbb"
output_dir.mkdir(parents=True, exist_ok=True)

output_file = output_dir / "tool_calling_trajectories.jsonl"

record = {
    "input": serialize_input_list(result["input"]),
    "output": serialize_input_list(result["output"]),
    "tools": TOOL_SCHEMAS,
    "reasoning_summaries": result["reasoning_summaries"],
    "usage": result["usage"],
}

with open(output_file, "a") as f:
    f.write(json.dumps(record) + "\n")

print(f"Saved trajectory to {output_file}")

## Test with a Few More Tickers

In [ ]:
test_tickers = [
    ("AAPL", "competitive position and market share"),
    ("TSLA", "recent news and analyst sentiment"),
    ("JPM", "financial health and balance sheet strength"),
]

for ticker, focus in test_tickers:
    print(f"\n{'='*60}")
    print(f"Researching {ticker} — {focus}")
    print(f"{'='*60}")
    
    res = run_tool_calling_agent(
        client=client,
        model=MODEL,
        user_prompt=f"Research {ticker} focusing on {focus}.",
        system_prompt=SYSTEM_PROMPT,
        reasoning_effort="medium",
    )
    
    # Count tool calls from the input list
    n_tool_results = sum(
        1 for item in res["input"]
        if isinstance(item, dict) and item.get("type") == "function_call_output"
    )
    memo_len = len(res["output_text"]) if res["output_text"] else 0
    print(f"  -> {n_tool_results} tool calls, {len(res['reasoning_summaries'])} reasoning steps, memo: {memo_len} chars")
    print(f"  -> Tokens: {res['usage']['input_tokens']} in / {res['usage']['output_tokens']} out / {res['usage']['reasoning_tokens']} reasoning")
    
    # Save
    record = {
        "input": serialize_input_list(res["input"]),
        "output": serialize_input_list(res["output"]),
        "tools": TOOL_SCHEMAS,
        "reasoning_summaries": res["reasoning_summaries"],
        "usage": res["usage"],
    }
    with open(output_file, "a") as f:
        f.write(json.dumps(record) + "\n")